In [ ]:
# !pip install numba
# !pip install pymap3d

In [ ]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
from eo_tools.util import show_cog
from eo_tools.S1_dev import fetch_dem_burst, geocode_burst, resample_burst_ampl
import numpy as np
from scipy.interpolate import griddata
from pathlib import Path
import logging
logging.basicConfig(level=logging.INFO)

In [ ]:
data_dir = "/data/S1"
master_dir = f"{data_dir}/S1A_IW_SLC__1SDV_20230903T183344_20230903T183412_050167_0609B4_100E.SAFE"
slave_dir = f"{data_dir}/S1A_IW_SLC__1SDV_20230915T183345_20230915T183413_050342_060F9F_85A4.SAFE"
master_re = "/data/res/master_re.tiff"
slave_re = "/data/res/slave_re.tiff"
burst_idx = 5
iw = 2
pol = "vv"

In [ ]:
file_dem = fetch_dem_burst(master_dir, iw=iw, pol=pol, burst_idx=burst_idx)

In [ ]:
az_mst, rg_mst, dem_prof = geocode_burst(
    master_dir, file_dem, iw=iw, pol=pol, burst_idx=burst_idx
)

In [ ]:
az_slv, rg_slv, dem_prof = geocode_burst(
    slave_dir, file_dem, iw=iw, pol=pol, burst_idx=burst_idx
)

In [ ]:
resample_burst_ampl(
    master_dir,
    master_re,
    az_mst,
    rg_mst,
    dem_prof,
    iw=iw,
    pol=pol,
    burst_idx=burst_idx,
    order=3,
)

In [ ]:
resample_burst_ampl(
    slave_dir,
    slave_re,
    az_slv,
    rg_slv,
    dem_prof,
    iw=iw,
    pol=pol,
    burst_idx=burst_idx,
    order=3,
)

In [ ]:
from folium import Map, LayerControl
m = Map()
show_cog(master_re, m, rescale="3,8.1", expression="log(b1)")
show_cog(slave_re, m, rescale="3,8.1", expression="log(b1)")
LayerControl().add_to(m)
m

In [ ]:
# interpolate azimuth and range to fixed grid
from eo_tools.S1_dev import read_metadata, read_chunk

dir_tiff_mst = Path(f"{master_dir}/measurement/")
dir_xml_mst = Path(f"{master_dir}/annotation/")
pth_xml_mst = dir_xml_mst.glob(f"*iw{iw}*{pol}*.xml")
pth_tiff_mst = dir_tiff_mst.glob(f"*iw{iw}*{pol}*.tiff")
pth_xml_mst = list(pth_xml_mst)[0]
pth_tiff_mst = list(pth_tiff_mst)[0]

meta_mst = read_metadata(pth_xml_mst)


dir_tiff_slv = Path(f"{slave_dir}/measurement/")
dir_xml_slv = Path(f"{slave_dir}/annotation/")
pth_xml_slv = dir_xml_slv.glob(f"*iw{iw}*{pol}*.xml")
pth_tiff_slv = dir_tiff_slv.glob(f"*iw{iw}*{pol}*.tiff")
pth_xml_slv = list(pth_xml_slv)[0]
pth_tiff_slv = list(pth_tiff_slv)[0]

meta_slv = read_metadata(pth_xml_slv)


In [ ]:
naz = int(meta_mst["product"]["swathTiming"]["linesPerBurst"])
nrg = int(meta_mst["product"]["swathTiming"]["samplesPerBurst"])
first_line = (burst_idx - 1) * naz


naz_slv = int(meta_slv["product"]["swathTiming"]["linesPerBurst"])
nrg_slv = int(meta_slv["product"]["swathTiming"]["samplesPerBurst"])
first_line_slv = (burst_idx - 1) * naz_slv

pixel spacing for IW:  
rg x az : 2.3 x 14.1 m
resize to get spacing comparable to DEM? -> much faster gridding

Method:  
1. regrid the data on a coarse grid using an irregular interpolator (coslty)
2. interpolate on a regular grid to match the size of the master 
3. warp the slave to the master's geometry

In [ ]:
# coarse grid to perform a first interpolation
rg_fixed = np.linspace(0,nrg-1,int(nrg/16))
az_fixed = np.linspace(0,naz-1,int(naz/8))

grg, gaz = np.meshgrid(rg_fixed, az_fixed)

In [ ]:
# discarding all pairs that cannot be mapped
cnd1 = (rg_mst >= 0) & (rg_mst < nrg)
cnd2 = (az_mst >= 0) & (az_mst < naz)
valid = cnd1 & cnd2 & (az_slv!=np.nan) & (rg_slv!=np.nan)

In [ ]:
coords = np.stack((az_mst[valid], rg_mst[valid])).T

In [ ]:
az_re = griddata(coords, az_slv[valid], (gaz.flat, grg.flat))
rg_re = griddata(coords, rg_slv[valid], (gaz.flat, grg.flat))

In [ ]:
from scipy.interpolate import interpn
grg2, gaz2 = np.meshgrid(np.arange(0,nrg), np.arange(0,naz))
az_w = interpn((az_fixed, rg_fixed), az_re.reshape((len(az_fixed), len(rg_fixed))), (gaz2.flat, grg2.flat))
rg_w = interpn((az_fixed, rg_fixed), rg_re.reshape((len(az_fixed), len(rg_fixed))), (gaz2.flat, grg2.flat))

In [ ]:
im_mst = read_chunk(pth_tiff_mst, first_line, naz)
im_slv = read_chunk(pth_tiff_slv, first_line_slv, naz_slv)

In [ ]:
from scipy.ndimage import map_coordinates
im_slv_re = map_coordinates(im_slv,(az_w, rg_w)).reshape((naz, nrg)) 

In [ ]:
phi = np.angle(im_slv_re*im_mst.conj())

In [ ]:
plt.figure(figsize=(20,20))
plt.imshow(phi[:,::8][:600,2000:3000], cmap="hsv")
# plt.figure(figsize=(20,20))
# plt.imshow(im_mst[:,::8], vmin=-np.pi, vmax=np.pi)

In [ ]:
# deramping

burst_meta = meta_slv["product"]["swathTiming"]["burstList"]["burst"][burst_idx-1]
az_time = burst_meta["azimuthTime"]
az_dt = meta_slv["product"]["imageAnnotation"]["azimuthTimeInterval"]

# az_time_mid = az_time + az_

In [ ]:
meta_slv["product"]["imageAnnotation"]["imageInformation"].keys()